# SLM Evaluation Results Analysis

This notebook will be used to analyze results from SLM runs.

In [1]:
import pandas as pd
import json

# Load the latest Qwen3-1.7B (1024 max tokens) samples file
samples_file = "RESULTS/llamacpp/Qwen__Qwen3-1.7B/samples_sms_extraction_2026-04-27T14-21-34.221880.jsonl"
df = pd.read_json(samples_file, lines=True)

# Display basic information about the dataframe
print(f"Number of samples: {len(df)}")
print("\nColumns:")
for col in df.columns:
    print(f"- {col}")

# Look at the first example's structure
print("\n\n--- Example 1 Structure ---")
example = df.iloc[0].to_dict()
for key, value in example.items():
    if isinstance(value, str):
        print(f"\n{key}:\n{value[:500]}..." if len(value) > 500 else f"\n{key}:\n{value}")
    else:
        print(f"\n{key}:\n{value}")

Number of samples: 406

Columns:
- doc_id
- doc
- target
- arguments
- resps
- filtered_resps
- filter
- metrics
- doc_hash
- prompt_hash
- target_hash
- json_validity
- rejection_accuracy
- field_accuracy
- amount_accuracy
- type_accuracy
- account_accuracy
- full_match_accuracy
- ghost_transaction_rate
- missed_transaction_rate
- date_accuracy
- merchant_accuracy
- few_shot_leakage_rate


--- Example 1 Structure ---

doc_id:
0

doc:
{'sms': 'Dear Customer, Your a/c no. XXXXXXXX6254 is debited for Rs.5000.00 on 02-05-22 and a/c XXXXXXX902 credited (IMPS Ref no 212219676001).If not done by you, call 1800111109 -SBI', 'sender': 'BP-SBIINB', 'expected': '{"amount": 5000.0, "merchant": null, "date": "02-05-2022", "type": "debit", "account": "a/c no. XXXXXXXX6254"}'}

target:
{"amount": 5000.0, "merchant": null, "date": "02-05-2022", "type": "debit", "account": "a/c no. XXXXXXXX6254"}

arguments:
{'gen_args_0': {'arg_0': '<|im_start|>user\nYou are extracting transaction details from an Ind

In [2]:
# Load both the 1024 token limit and 4096 token limit runs
samples_1024 = "RESULTS/llamacpp/Qwen__Qwen3-1.7B/samples_sms_extraction_2026-04-27T14-21-34.221880.jsonl"
samples_4096 = "RESULTS/llamacpp/Qwen__Qwen3-1.7B/samples_sms_extraction_2026-04-27T14-41-47.716490.jsonl"

df_1024 = pd.read_json(samples_1024, lines=True)
df_4096 = pd.read_json(samples_4096, lines=True)

# Merge the two dataframes on doc_id to align the responses for each SMS
# the 'resps' column contains raw responses including thinking blocks
# the 'filtered_resps' column contains the JSON extracted output
comparison_df = pd.merge(
    df_1024[['doc_id', 'doc', 'target', 'resps', 'filtered_resps']],
    df_4096[['doc_id', 'resps', 'filtered_resps']],
    on='doc_id',
    suffixes=('_1024', '_4096')
)

# Look at the first sample's raw responses
sample = comparison_df.iloc[0]
print("--- SMS Document ---")
print(sample['doc']['sms'])
print("\n--- Ground Truth Target ---")
print(sample['target'])
print("\n--- 1024 Max Tokens Response (First 500 chars) ---")
resp_1024 = sample['resps_1024'][0]
print(resp_1024[:500] + "..." if len(resp_1024) > 500 else resp_1024)
print("\n--- 4096 Max Tokens Response (First 500 chars) ---")
resp_4096 = sample['resps_4096'][0]
print(resp_4096[:500] + "..." if len(resp_4096) > 500 else resp_4096)


--- SMS Document ---
Dear Customer, Your a/c no. XXXXXXXX6254 is debited for Rs.5000.00 on 02-05-22 and a/c XXXXXXX902 credited (IMPS Ref no 212219676001).If not done by you, call 1800111109 -SBI

--- Ground Truth Target ---
{"amount": 5000.0, "merchant": null, "date": "02-05-2022", "type": "debit", "account": "a/c no. XXXXXXXX6254"}

--- 1024 Max Tokens Response (First 500 chars) ---
['<think>\nOkay, let\'s tackle this problem step by step. The user provided an SMS and wants me to extract transaction details based on the given rules. Let me start by analyzing the sender ID and the content of the SMS.\n\nThe sender is BP-SBIINB. The format is XX-YYYYYY, where the first two characters are the operator prefix. Here, BP is the operator, and SBIINB is the entity code. For Indian banks, the entity codes should resemble their names or acronyms. SBIINB is indeed for State Bank of India (SBI), so this looks like a real bank sender.\n\nNow, checking the content: "Dear Customer, Your a/c no. XXX

Available Metrics:
- json_validity
- rejection_accuracy
- field_accuracy
- amount_accuracy
- type_accuracy
- account_accuracy
- full_match_accuracy
- ghost_transaction_rate
- missed_transaction_rate
- date_accuracy
- merchant_accuracy
- few_shot_leakage_rate

--- Example Sample Metrics Dictionary ---
[
  "json_validity",
  "rejection_accuracy",
  "field_accuracy",
  "amount_accuracy",
  "type_accuracy",
  "account_accuracy",
  "full_match_accuracy",
  "ghost_transaction_rate",
  "missed_transaction_rate",
  "date_accuracy",
  "merchant_accuracy",
  "few_shot_leakage_rate"
]


In [6]:
df_1024[[
  "json_validity",
  "rejection_accuracy",
  "field_accuracy",
  "amount_accuracy",
  "type_accuracy",
  "account_accuracy",
  "full_match_accuracy",
  "ghost_transaction_rate",
  "missed_transaction_rate",
  "date_accuracy",
  "merchant_accuracy",
  "few_shot_leakage_rate"
]].describe()

,json_validity,rejection_accuracy,field_accuracy,amount_accuracy,type_accuracy,account_accuracy,full_match_accuracy,ghost_transaction_rate,missed_transaction_rate,date_accuracy,merchant_accuracy,few_shot_leakage_rate
count,406.0,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.00000,406.000000,406.000000,406.000000,406.0
mean,1.0,0.899015,0.805419,0.899015,0.899015,0.899015,0.766010,0.03202,0.068966,0.834975,0.830049,0.0
std,0.0,0.301681,0.311817,0.301681,0.301681,0.301681,0.423888,0.17627,0.253708,0.371661,0.376053,0.0
min,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0
25%,1.0,1.000000,0.750000,1.000000,1.000000,1.000000,1.000000,0.00000,0.000000,1.000000,1.000000,0.0
50%,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.00000,0.000000,1.000000,1.000000,0.0
75%,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.00000,0.000000,1.000000,1.000000,0.0
max,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,0.0
